In [1]:
import librosa as lb
import numpy as np
import pickle
from eval_tools import getGroundTruthTimestamps
import utils.constants as constants
import plotly.graph_objs as go

In [10]:
s = 5

dtw_hyp_file = f"experiments/DTW/s{s}/hyp.npy"
noa_hyp_file = f"experiments/NOA/s{s}/hyp.npy"
noa_tsm_file = f"experiments/NOA/s{s}/tsm.npy"

# load
dtw_hyp = np.load(dtw_hyp_file)
noa_hyp = np.load(noa_hyp_file)
noa_tsm = np.load(noa_tsm_file)

# load the ground truth timestamps
query_annot_file = f'scenarios/s{s}/query.beats'
ref_annot_file = f'scenarios/s{s}/ref.beats'
gt = getGroundTruthTimestamps(query_annot_file, ref_annot_file).T

In [11]:
pair_txt_file = f'scenarios/s{s}/pair.txt'
pair_txt = open(pair_txt_file, 'r').readlines()
pair_txt = [line.strip().split() for line in pair_txt]
id1 = pair_txt[0][0]
id2 = pair_txt[0][1]

# load the chroma features
audio_path_1 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id1}.wav'
audio_path_2 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id2}.wav'

# load the features
f1 = np.load(f'features/chroma_stft_norm2/{id1}.npy')
f2 = np.load(f'features/chroma_stft_norm2/{id2}.npy')

In [12]:
from noa_kalman import alignNOAKalman
Q = np.array([[1e-2, 0], [0, 1e-4]])
R = np.array([[30]])
sigma_x = 2
sigma_v = 0.004
results = alignNOAKalman(f1, f2, Q = Q, R = R, sigma_x = sigma_x, sigma_v = sigma_v, return_path = False, onset_reduction_factor=1)
results100 = alignNOAKalman(f1, f2, Q = Q, R = R, sigma_x = sigma_x, sigma_v = sigma_v, return_path = False, onset_reduction_factor=100)

In [ ]:
# plot results velocity over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.arange(0, len(results.velocity_history)) * 512/22050, y=results.velocity_history, mode='lines', name='NOA Kalman'
))
fig.update_layout(
    title='Velocity History',
    xaxis_title='Time (s)',
    yaxis_title='Relative Tempo',
    width=900,
    height=400
)
fig.show()

In [ ]:
# plot results innovation over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.arange(0, len(results.innovation_history)) * 512/22050, y=np.asarray(results.innovation_history) * 512/22050, mode='lines', name='NOA Kalman'
))
fig.update_layout(
    title='Innovation History',
    xaxis_title='Time (s)',
    yaxis_title='Innovation (s)',
    width=900,
    height=400
)
fig.show()

In [ ]:
# plot the P matrix [0][0], [1][1] over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.arange(0, len(results.P_history)) * 512/22050, y=np.asarray(results.P_history)[:, 1,1], mode='lines', name='NOA Kalman'
))
fig.update_layout(
    title='P Matrix [0][0] over Time',
    xaxis_title='Time (s)',
    yaxis_title='P [0][0]',
    width=900,
    height=400
)
fig.show()

In [14]:
fig = go.Figure()
fig.update_layout(
    title='Alignment and TSM Paths',
    xaxis_title='Query Time (s)',
    yaxis_title='Reference Time (s)',
    width=900,
    height=700
)
fig.add_trace(go.Scatter(
    x=results.get_noa_path()[0, :], y=results.get_noa_path()[1, :], mode='lines', name='NOA Hyp'
))
fig.add_trace(go.Scatter(
    x=results.get_path()[0, :], y=results.get_path()[1, :], mode='lines', name='NOA Kalman'
))
fig.add_trace(go.Scatter(
    x=results100.get_path()[0, :], y=results100.get_path()[1, :], mode='lines', name='NOA Kalman 100'
))
fig.add_trace(go.Scatter(
    x=gt[0], y=gt[1], mode='markers', name='Ground Truth'
))
# Check bounds of noa_hyp[0] and [1] for diagonal plotting
x_min = min(np.min(noa_hyp[0]), np.min(noa_hyp[1]))
x_max = max(np.max(noa_hyp[0]), np.max(noa_hyp[1]))
fig.add_trace(go.Scatter(
    x=[x_min, x_max],
    y=[x_min, x_max],
    mode='lines',
    name='Diagonal',
    line=dict(dash='dash', color='grey')
))

fig.show()

In [ ]:
# innovations_diagnostics.py
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

def innovation_diagnostics(innov, S=None, verbose=True):
    """
    innov : 1D array of innovations y_k (observed - predicted observation)
    S     : None or 1D array of predicted innovation variances (scalar obs case)
    Returns a dict of diagnostics and makes plots.
    """
    innov = np.asarray(innov).flatten()
    N = len(innov)
    mean_raw = np.mean(innov)
    var_raw = np.var(innov, ddof=1)
    std_raw = np.sqrt(var_raw)

    # If predicted innovation variance S is supplied, standardize per-sample.
    if S is None:
        S = np.ones(N) * var_raw  # fallback: use sample var so standardized has var~1
    else:
        S = np.asarray(S).flatten()
        assert len(S) == N, "S must have same length as innovations"

    z = innov / np.sqrt(S)   # standardized innovations (scalar observation case)
    z_mean = np.mean(z)
    z_var = np.var(z, ddof=1)
    outlier_rate = np.mean(np.abs(z) > 3) * 100.0

    # Tests
    # 1) t-test for raw mean = 0 (student t using sample std)
    se_raw = std_raw / np.sqrt(N)
    t_raw = mean_raw / se_raw
    p_raw = 2 * (1 - stats.t.cdf(abs(t_raw), df=N-1))

    # 2) t-test for standardized mean = 0
    se_z = np.std(z, ddof=1) / np.sqrt(N)
    t_z = z_mean / se_z
    p_z = 2 * (1 - stats.t.cdf(abs(t_z), df=N-1))

    # 3) Normality test (Jarque-Bera)
    jb_stat, jb_p = stats.jarque_bera(z)

    # 4) Ljung-Box for whiteness (use lags up to e.g. 40 or  min(40, N/5))
    lb_lags = min(40, max(5, int(N / 50)))
    lb_res = acorr_ljungbox(z, lags=[lb_lags], return_df=True)

    # 5) NIS (if S provided): sum z^2 -> should be chi-square with df=N
    nis = np.sum(z**2)
    nis_p = 1 - stats.chi2.cdf(nis, df=N)  # extremely small p => inconsistent

    results = {
        'N': N,
        'mean_raw': mean_raw,
        'std_raw': std_raw,
        'var_raw': var_raw,
        't_raw': t_raw,
        'p_raw': p_raw,
        'z_mean': z_mean,
        'z_var': z_var,
        't_z': t_z,
        'p_z': p_z,
        'outlier_rate_pct': outlier_rate,
        'jarque_bera_stat': jb_stat,
        'jarque_bera_p': jb_p,
        'ljung_box': lb_res,
        'nis': nis,
        'nis_p': nis_p
    }

    if verbose:
        print("=== Innovation Diagnostics ===")
        print(f"Count: {N}")
        print(f"Mean:  {mean_raw:.6f}")
        print(f"Var:   {var_raw:.6f}")
        print(f"Std:   {std_raw:.6f}\n")
        print("Standardized innovations:")
        print(f"  Mean: {z_mean:.6f}")
        print(f"  Var:  {z_var:.6f}")
        print(f"  Outlier rate |z|>3: {outlier_rate:.2f}%\n")
        print("Statistical tests:")
        print(f"  Raw mean t-stat = {t_raw:.3f}, p = {p_raw:.4g}")
        print(f"  Std mean t-stat = {t_z:.3f}, p = {p_z:.4g}")
        print(f"  Jarque-Bera stat = {jb_stat:.3f}, p = {jb_p:.4g}")
        print(f"  Ljung-Box (lag {lb_lags}) p =\n{lb_res}\n")
        print(f"  NIS = {nis:.3f}  (chi2 p-value = {nis_p:.4g})")

    # Plots
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.hist(z, bins=60, density=True, alpha=0.6, label='standardized innovations')
    xs = np.linspace(-6,6,501)
    plt.plot(xs, stats.norm.pdf(xs,0,1), label='N(0,1) pdf')
    plt.legend()
    plt.title("Histogram of standardized innovations")
    plt.subplot(1,2,2)
    stats.probplot(z, dist="norm", plot=plt)
    plt.title("QQ-plot")
    plt.tight_layout()
    plt.show()

    # ACF
    nlags = min(200, int(N/4))
    acf_vals = acf(z, nlags=nlags, fft=True)
    results['acf_vals'] = acf_vals
    plt.figure(figsize=(9,3))
    plt.stem(np.arange(len(acf_vals)), acf_vals)
    plt.xlabel("lag")
    plt.ylabel("ACF")
    plt.title("Autocorrelation of standardized innovations")
    plt.show()

    # running variance vs predicted S
    window = max(1, int(N/100))
    running_var = np.array([np.var(innov[max(0,i-window):i+1], ddof=1) for i in range(N)])
    plt.figure(figsize=(9,3))
    plt.plot(running_var, label='running sample variance of innov (window {})'.format(window))
    plt.plot(S, label='predicted innovation variance S_k', alpha=0.7)
    plt.legend()
    plt.title("Running variance vs predicted S")
    plt.show()

    return results

# Usage:
# diagnostics = innovation_diagnostics(my_innov_array, S=my_predicted_S_array)


In [ ]:
innovation_diagnostics(results.innovation_history, results.innovation_variance_history)